# Campus mobility prediction

Run cells from top to bottom with the project's `.venv` Python kernel. Paths are relative
to this project folder. This notebook generates the dataset, checks Person 1's work,
and runs Person 2's GRU experiments.

Training uses five consecutive readings to predict the serving cell three seconds ahead.
Fine-tuning and scratch training use the same first 10, 25, and 50 morning trips.
Checkpoints and prediction CSVs are saved in `artifacts/`.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import random
import tempfile
import unittest
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import Markdown, display


## Experiment settings

Edit these values before running the training cells. The default run uses CPU, seed 42,
and validation loss to select checkpoints. The final test trips are used after training.


In [2]:
data_dir = Path("data")
output_dir = Path("artifacts")
seed = 42
hidden_size = 32
pretrain_epochs = 20
adapt_epochs = 40
patience = 8
batch_size = 512
learning_rate = 0.001
threads = 2

torch.set_num_threads(threads)
output_dir.mkdir(parents=True, exist_ok=True)


## Person 1's campus and trips

The campus has five places and four base stations. Cell IDs follow the base-station order.
Each reading contains `trip_id, time_s, x_m, y_m, speed_m_s, heading_rad, serving_cell`.
Trip IDs are local to each CSV. Cell labels use true positions while x/y readings contain noise.


In [3]:
PLACES = {
    "hostel":    (50, 50),
    "library":   (350, 80),
    "lecture_a": (230, 290),
    "lecture_b": (340, 330),
    "canteen":   (60, 300),
}

BASE_STATIONS = [(100, 100), (300, 100), (100, 300), (300, 300)]

NOISE_M = 3.0

SPLITS = [
    # name,            scenario,  n_trips, seed
    ("general_train", "general", 300, 1),
    ("general_val",   "general", 50,  2),
    ("morning_adapt", "morning", 50,  3),
    ("morning_val",   "morning", 30,  4),
    ("morning_test",  "morning", 100, 5),
]


In [4]:
def serving_cell(x, y):
    """Closest base station wins."""
    return min(range(len(BASE_STATIONS)),
               key=lambda i: (x - BASE_STATIONS[i][0]) ** 2 + (y - BASE_STATIONS[i][1]) ** 2)

def pick_stops(scenario, rng):
    names = list(PLACES)
    if scenario == "morning":  # hostel -> mostly a lecture hall
        dest = rng.choices(["lecture_a", "lecture_b", "library", "canteen"],
                           weights=[4, 4, 1, 1])[0]
        return ["hostel", dest]
    stops = [rng.choice(names)]  # general: random start, 3 random legs
    for _ in range(3):
        stops.append(rng.choice([n for n in names if n != stops[-1]]))
    return stops


In [5]:
def make_trip(trip_id, scenario, rng, dt=1.0):
    """Return rows for one trip, one row every dt seconds."""
    stops = pick_stops(scenario, rng)
    speed = rng.uniform(1.0, 1.6)  # m/s, constant per trip
    x, y = PLACES[stops[0]]
    heading = 0.0
    rows = []

    def record(v):
        # position is noisy (GPS-like); the serving cell uses the true position
        rows.append([trip_id, round(len(rows) * dt, 3),
                     round(x + rng.gauss(0, NOISE_M), 2), round(y + rng.gauss(0, NOISE_M), 2),
                     v, round(heading, 3), serving_cell(x, y)])

    def walk(tx, ty):
        nonlocal x, y, heading
        heading = math.atan2(ty - y, tx - x)
        while math.hypot(tx - x, ty - y) > speed * dt:
            record(round(speed, 2))
            x += speed * dt * math.cos(heading)
            y += speed * dt * math.sin(heading)
        x, y = tx, ty

    for i, name in enumerate(stops[1:], start=1):
        tx, ty = PLACES[name]
        # one random via-point, so the same origin and destination give different routes
        side = math.atan2(ty - y, tx - x) + math.pi / 2
        off = rng.uniform(-40, 40)
        walk((x + tx) / 2 + off * math.cos(side), (y + ty) / 2 + off * math.sin(side))
        walk(tx, ty)
        if i < len(stops) - 1:  # pause at intermediate stops
            for _ in range(int(rng.randint(0, 20) / dt)):
                record(0.0)
    record(0.0)  # final arrival
    return rows


In [6]:
def make_dataset(scenario, n_trips, seed, dt=1.0):
    rng = random.Random(seed)
    return [r for i in range(n_trips) for r in make_trip(i, scenario, rng, dt)]

def save(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["trip_id", "time_s", "x_m", "y_m", "speed_m_s", "heading_rad", "serving_cell"])
        w.writerows(rows)


### Generate the five CSV files

This cell regenerates all five splits using their fixed seeds. Training windows are created
later, within individual trips.


In [7]:
for name, scenario, count, split_seed in SPLITS:
    rows = make_dataset(scenario, count, split_seed)
    save(rows, str(data_dir / f"{name}.csv"))
    print(f"{name}: {count} trips, {len(rows)} readings")


general_train: 300 trips, 193940 readings


general_val: 50 trips, 33144 readings
morning_adapt: 50 trips, 13260 readings


morning_val: 30 trips, 8044 readings


morning_test: 100 trips, 26024 readings


## Verify the dataset

Check the schema, timestamps, cell IDs, whole trips, duplicate trips, and exact regeneration
from the documented seeds. The report also counts handovers and changed-target windows.

Handovers have one-second resolution. Generating at `dt=0.1` consumes random numbers
differently and changes the data. Matching sub-second events and the campus map demo remain
pending in Person 1's work.


In [8]:
COLUMNS = ["trip_id", "time_s", "x_m", "y_m", "speed_m_s",
           "heading_rad", "serving_cell"]


In [9]:
def read_trips(path):
    """Read ordered, one-second trips. IDs are local to each CSV file."""
    trips = {}
    previous_id = None
    with Path(path).open(newline="", encoding="utf-8") as source:
        reader = csv.DictReader(source)
        if reader.fieldnames != COLUMNS:
            raise ValueError(f"{path}: expected columns {COLUMNS}")
        for line, row in enumerate(reader, 2):
            try:
                trip_id = int(row["trip_id"])
                values = [float(row[key]) for key in COLUMNS[1:]]
            except (ValueError, TypeError) as error:
                raise ValueError(f"{path}:{line}: invalid numeric row") from error
            if not all(math.isfinite(value) for value in values):
                raise ValueError(f"{path}:{line}: non-finite reading")
            time, _, _, speed, heading, cell = values
            if cell != int(cell) or not 0 <= cell < len(BASE_STATIONS):
                raise ValueError(f"{path}:{line}: invalid cell ID")
            if speed < 0 or abs(heading) > math.pi + 0.001:
                raise ValueError(f"{path}:{line}: invalid speed or heading")
            if trip_id != previous_id and trip_id in trips:
                raise ValueError(f"{path}:{line}: interleaved trip")
            readings = trips.setdefault(trip_id, [])
            expected_time = readings[-1][0] + 1 if readings else 0
            if not math.isclose(time, expected_time, abs_tol=1e-6):
                raise ValueError(f"{path}:{line}: expected time {expected_time}, got {time}")
            readings.append(values)
            previous_id = trip_id
    if not trips or any(len(rows) < 8 for rows in trips.values()):
        raise ValueError(f"{path}: each trip needs at least 8 readings")
    return trips


In [10]:
def audit(data_dir):
    summaries = {}
    seen = {}
    for name, scenario, expected_count, seed in SPLITS:
        path = Path(data_dir) / f"{name}.csv"
        trips = read_trips(path)
        if sorted(trips) != list(range(expected_count)):
            raise ValueError(f"{path}: expected trip IDs 0 through {expected_count - 1}")
        # Compare content, since an ID such as 0 occurs in every split.
        for trip_id, rows in trips.items():
            digest = hashlib.sha256(json.dumps(rows).encode()).hexdigest()
            if digest in seen:
                raise ValueError(f"Duplicate trip in {name}/{trip_id} and {seen[digest]}")
            seen[digest] = f"{name}/{trip_id}"
        regenerated = make_dataset(scenario, expected_count, seed)
        actual = [[trip_id, *row] for trip_id, rows in trips.items() for row in rows]
        if actual != regenerated:
            raise ValueError(f"{path}: data differs from its documented seed/configuration")
        handovers = sum(a[-1] != b[-1] for rows in trips.values()
                        for a, b in zip(rows, rows[1:]))
        windows = sum(len(rows) - 7 for rows in trips.values())
        changed = sum(rows[end][-1] != rows[end + 3][-1]
                      for rows in trips.values() for end in range(4, len(rows) - 3))
        summaries[name] = {"trips": len(trips), "rows": len(actual),
                           "handovers": handovers, "windows": windows,
                           "changed_windows": changed,
                           "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    return summaries


In [11]:
audit_report = audit(data_dir)
print(json.dumps(audit_report, indent=2))

data = {name: read_trips(data_dir / f"{name}.csv") for name in audit_report}


{
  "general_train": {
    "trips": 300,
    "rows": 193940,
    "handovers": 1046,
    "windows": 191840,
    "changed_windows": 3130,
    "sha256": "3bd20fe98a8c296ce44a8587788443a8f6a0f1d58d8f5c3a99324ff83d4c711e"
  },
  "general_val": {
    "trips": 50,
    "rows": 33144,
    "handovers": 185,
    "windows": 32794,
    "changed_windows": 555,
    "sha256": "d89a41477aa8505784ec01cd5fe792505773a2150be680e1e62ec58e7a32bb64"
  },
  "morning_adapt": {
    "trips": 50,
    "rows": 13260,
    "handovers": 94,
    "windows": 12910,
    "changed_windows": 280,
    "sha256": "7f3dfe1f2de763a6d79f381cc178e6c2bd2ab78dee677fc5e4fc6d3b625b7521"
  },
  "morning_val": {
    "trips": 30,
    "rows": 8044,
    "handovers": 53,
    "windows": 7834,
    "changed_windows": 159,
    "sha256": "5bbe19941dd7da388b092a0a3236e4366aeb0601388e5d095afe82c9d3fb1842"
  },
  "morning_test": {
    "trips": 100,
    "rows": 26024,
    "handovers": 176,
    "windows": 25324,
    "changed_windows": 528,
    "sha256"

## Prepare five-reading inputs

Features are campus-scaled x/y, scaled speed, sine/cosine of heading, and a one-hot current
cell. Fixed scaling uses no statistics from validation or test data.
For readings at seconds 8 through 12, the label is the cell at second 15.


In [12]:
WINDOW = 5

HORIZON = 3

N_CELLS = len(BASE_STATIONS)

FEATURES = ["x/400", "y/400", "speed/1.6", "sin(heading)", "cos(heading)",
            *[f"cell_{cell}" for cell in range(N_CELLS)]]

def seed_everything(seed):
    # Training only draws random numbers through PyTorch.
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)

def encode(readings):
    """Convert CSV readings into the nine model features."""
    rows = np.asarray(readings, dtype=np.float32)
    if rows.ndim != 2 or rows.shape[1] != 6 or not np.isfinite(rows).all():
        raise ValueError("Readings must be finite [time, x, y, speed, heading, cell] rows")
    cells = rows[:, 5]
    if np.any(cells != cells.astype(int)) or np.any((cells < 0) | (cells >= N_CELLS)):
        raise ValueError("Cell IDs must be integers in 0..3")
    x = rows[:, 1] / 400
    y = rows[:, 2] / 400
    speed = rows[:, 3] / 1.6
    heading = rows[:, 4]
    cell_features = np.eye(N_CELLS, dtype=np.float32)[cells.astype(int)]
    return np.column_stack((x, y, speed, np.sin(heading), np.cos(heading), cell_features))


In [13]:
@dataclass
class Windows:
    x: torch.Tensor
    y: torch.Tensor
    current: np.ndarray
    trip_ids: np.ndarray
    times: np.ndarray

def make_windows(trips, limit=None):
    """Select whole trips before windowing. Every target is exactly t + 3."""
    selected = sorted(trips)
    if limit is not None:
        selected = selected[:limit]
    if not selected or (limit is not None and len(selected) != limit):
        raise ValueError(f"Requested {limit} trips, available {len(trips)}")
    inputs, targets, current, trip_ids, times = [], [], [], [], []
    for trip_id in selected:
        rows = np.asarray(trips[trip_id], dtype=np.float32)
        if len(rows) < WINDOW + HORIZON:
            raise ValueError(f"Trip {trip_id} is too short")
        features = encode(rows)
        for end in range(WINDOW - 1, len(rows) - HORIZON):
            inputs.append(features[end - WINDOW + 1:end + 1])
            targets.append(int(rows[end + HORIZON, 5]))
            current.append(int(rows[end, 5]))
            times.append(rows[end, 0])
            trip_ids.append(trip_id)
    return Windows(
        x=torch.from_numpy(np.stack(inputs)),
        y=torch.tensor(targets, dtype=torch.long),
        current=np.asarray(current),
        trip_ids=np.asarray(trip_ids),
        times=np.asarray(times),
    )


In [14]:
general_train = make_windows(data["general_train"])
general_val = make_windows(data["general_val"])
morning_val = make_windows(data["morning_val"])
print("General training inputs", tuple(general_train.x.shape))


General training inputs (191840, 5, 9)


## GRU and training

A single GRU layer reads each window. A linear layer produces four cell scores.
Training uses cross-entropy, Adam, and early stopping on validation loss.
Inference converts the scores into four probabilities using softmax.


In [15]:
class MobilityGRU(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.gru = nn.GRU(len(FEATURES), hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, N_CELLS)

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.classifier(hidden[-1])

@torch.inference_mode()
def probabilities(model, x, batch_size=2048):
    model.eval()
    return np.concatenate([model(batch).softmax(-1).numpy()
                           for batch in x.split(batch_size)])

def metrics(probs, windows):
    labels = windows.y.numpy()
    predicted = probs.argmax(axis=1)
    changed = labels != windows.current
    correct = predicted == labels
    target_probabilities = probs[np.arange(len(labels)), labels]
    cross_entropy = -np.log(np.maximum(target_probabilities, 1e-12)).mean()
    return {
        "windows": len(labels),
        "changed_windows": int(changed.sum()),
        "accuracy": float(correct.mean()),
        "changed_accuracy": float(correct[changed].mean()) if changed.any() else None,
        "unchanged_accuracy": float(correct[~changed].mean()) if (~changed).any() else None,
        "cross_entropy": float(cross_entropy),
    }


In [16]:
def fit(model, train, validation, epochs, learning_rate, batch_size, patience, seed):
    seed_everything(seed)
    loader = DataLoader(TensorDataset(train.x, train.y), batch_size=batch_size,
                        shuffle=True, generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for x, y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(y)
        scores = metrics(probabilities(model, validation.x), validation)
        history.append({"epoch": epoch, "train_loss": total_loss / len(train.y), **scores})
        if scores["cross_entropy"] < best_loss:
            best_loss = scores["cross_entropy"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
        if epoch == 1 or epoch % 5 == 0:
            print(f"  epoch {epoch:2}: val loss={scores['cross_entropy']:.4f}", flush=True)
        if epoch - best_epoch >= patience:
            break
    model.load_state_dict(best_state)
    return {"best_epoch": best_epoch, "history": history}


In [17]:
def save_checkpoint(model, path, hidden_size, seed, training):
    torch.save({"state_dict": model.state_dict(), "hidden_size": hidden_size,
                "features": FEATURES, "window": WINDOW, "horizon_s": HORIZON,
                "n_cells": N_CELLS, "seed": seed, "training": training}, path)

class Predictor:
    """Person 3/4 interface. predict_proba returns probabilities in cell-ID order.

    Pass exactly five ordered readings as dictionaries with Person 1's CSV keys.
    Use each prediction at the timestamp of the last reading in that window.
    """
    def __init__(self, checkpoint):
        saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
        if (saved["features"] != FEATURES or saved["window"] != WINDOW
                or saved["horizon_s"] != HORIZON or saved["n_cells"] != N_CELLS):
            raise ValueError("Checkpoint uses an incompatible prediction interface")
        self.model = MobilityGRU(saved["hidden_size"])
        self.model.load_state_dict(saved["state_dict"])

    def predict_proba(self, readings):
        if len(readings) != WINDOW:
            raise ValueError("Provide exactly five readings")
        keys = ["time_s", "x_m", "y_m", "speed_m_s", "heading_rad", "serving_cell"]
        rows = np.asarray([[float(row[key]) for key in keys] for row in readings], dtype=np.float32)
        if not np.allclose(np.diff(rows[:, 0]), 1.0):
            raise ValueError("Readings must be consecutive, one second apart")
        ids = [row.get("trip_id") for row in readings]
        if len(set(ids)) != 1:
            raise ValueError("Readings must belong to one trip")
        return probabilities(self.model, torch.from_numpy(encode(rows)[None]))[0]

def export_predictions(path, probs, windows):
    # Future labels are kept in metrics/demo only. Policy inputs contain current data.
    with path.open("w", newline="", encoding="utf-8") as destination:
        writer = csv.writer(destination)
        writer.writerow(["trip_id", "time_s", "serving_cell", "target_time_s",
                         *[f"p_cell_{cell}" for cell in range(N_CELLS)]])
        for i, vector in enumerate(probs):
            writer.writerow([int(windows.trip_ids[i]), float(windows.times[i]),
                             int(windows.current[i]), float(windows.times[i] + HORIZON),
                             *vector.tolist()])


### Pre-train on general trips

Use `general_val` to select the pre-training checkpoint.


In [18]:
seed_everything(seed)
pretrained = MobilityGRU(hidden_size)
training = fit(pretrained, general_train, general_val, pretrain_epochs,
               learning_rate, batch_size, patience, seed)
save_checkpoint(pretrained, output_dir / "pretrained.pt", hidden_size, seed, training)
runs = {"pretrained": training}


  epoch  1: val loss=0.0786


  epoch  5: val loss=0.0386


  epoch 10: val loss=0.0277


  epoch 15: val loss=0.0227


  epoch 20: val loss=0.0257


### Fine-tune and train from scratch

Each fine-tuning run starts from the same general checkpoint. Scratch models start with
fresh weights. Both use the same morning trips, learning rate, epoch limit, and validation
set. The smaller adaptation sets are contained within the larger sets.


In [19]:
for count in (10, 25, 50):
    adaptation = make_windows(data["morning_adapt"], count)
    for method in ("finetuned", "scratch"):
        name = f"{method}_{count}"
        seed_everything(seed)
        model = MobilityGRU(hidden_size)
        if method == "finetuned":
            model.load_state_dict(pretrained.state_dict())
        print(f"Training {name}", flush=True)
        training = fit(model, adaptation, morning_val, adapt_epochs,
                       learning_rate, batch_size, patience, seed)
        save_checkpoint(model, output_dir / f"{name}.pt", hidden_size, seed, training)
        runs[name] = training


Training finetuned_10


  epoch  1: val loss=0.0388


  epoch  5: val loss=0.0267


  epoch 10: val loss=0.0263


  epoch 15: val loss=0.0286


  epoch 20: val loss=0.0249


... (34 outputs left out)

  epoch 25: val loss=0.0676


  epoch 30: val loss=0.0559


  epoch 35: val loss=0.0516


  epoch 40: val loss=0.0467


## Evaluate frozen checkpoints

Evaluate every model on the same held-out morning windows. Changed-target accuracy counts
only windows where the cell three seconds ahead differs from the current cell.
Overall accuracy is also reported because most windows remain in the current cell.


In [20]:
test = make_windows(data["morning_test"])
comparison = {}
for name in runs:
    model = Predictor(output_dir / f"{name}.pt").model
    probs = probabilities(model, test.x)
    comparison[name] = metrics(probs, test)
    export_predictions(output_dir / f"{name}_predictions.csv", probs, test)

persistence = np.eye(N_CELLS, dtype=np.float32)[test.current]
comparison["current_cell_baseline"] = metrics(persistence, test)

table = "| Model | Overall accuracy | Changed-target accuracy |\n|---|---:|---:|\n"
for name, scores in comparison.items():
    table += f"| {name} | {scores['accuracy']:.1%} | {scores['changed_accuracy']:.1%} |\n"
display(Markdown(table))


| Model | Overall accuracy | Changed-target accuracy |
|---|---:|---:|
| pretrained | 98.8% | 76.3% |
| finetuned_10 | 98.9% | 68.2% |
| scratch_10 | 97.2% | 0.0% |
| finetuned_25 | 98.6% | 64.2% |
| scratch_25 | 97.7% | 0.0% |
| finetuned_50 | 98.8% | 55.9% |
| scratch_50 | 98.2% | 30.1% |
| current_cell_baseline | 97.9% | 0.0% |


In [21]:
config = {
    "data_dir": str(data_dir),
    "output_dir": str(output_dir),
    "seed": seed,
    "hidden_size": hidden_size,
    "pretrain_epochs": pretrain_epochs,
    "adapt_epochs": adapt_epochs,
    "patience": patience,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "threads": threads,
}
report = {
    "config": config,
    "torch_version": str(torch.__version__),
    "numpy_version": np.__version__,
    "dataset": audit_report,
    "training": runs,
    "test": comparison,
}
(output_dir / "results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved checkpoints, predictions, and results in", output_dir)


Saved checkpoints, predictions, and results in artifacts


## Held-out trip demo

Edit `trip_id` and `adaptation_trips`, then rerun this cell to compare checkpoints on another
trip or budget. Every changed-target window is shown, including incorrect predictions.
Fine-tuning may improve or reduce prediction quality for an individual trip.


In [22]:
trip_id = 0
adaptation_trips = 10

trips = read_trips(data_dir / "morning_test.csv")
if trip_id not in trips:
    raise ValueError(f"Unknown test trip {trip_id}")
windows = make_windows({trip_id: trips[trip_id]})
names = ["pretrained", f"scratch_{adaptation_trips}", f"finetuned_{adaptation_trips}"]
probs = {name: probabilities(Predictor(output_dir / f"{name}.pt").model, windows.x)
         for name in names}
print(f"Held-out trip {trip_id}, training budget {adaptation_trips} morning trips")
print("time  current  future  pretrained  scratch  finetuned  P(future) before -> after")
# Display every changed-target window in this trip, including wrong predictions.
for i in np.flatnonzero(windows.current != windows.y.numpy()):
    target = int(windows.y[i])
    predictions = [int(probs[name][i].argmax()) for name in names]
    before, after = probs[names[0]][i, target], probs[names[-1]][i, target]
    print(f"{windows.times[i]:4.0f}  {windows.current[i]:7}  {target:6}  "
          f"{predictions[0]:10}  {predictions[1]:7}  {predictions[2]:9}  {before:.3f} -> {after:.3f}")


Held-out trip 0, training budget 10 morning trips
time  current  future  pretrained  scratch  finetuned  P(future) before -> after
 132        0       2           2        0          2  0.863 -> 0.761
 133        0       2           2        0          2  0.983 -> 0.973
 134        0       2           2        0          2  0.961 -> 0.933
 156        2       3           3        2          3  0.574 -> 0.569
 157        2       3           3        2          3  0.659 -> 0.653
 158        2       3           3        2          3  0.604 -> 0.598


## Prediction interface for Persons 3 and 4

After running the definition cells above, `Predictor.predict_proba()` accepts five consecutive
reading dictionaries from one trip. Its four output entries correspond to cell IDs 0 through 3
and predict the serving cell three seconds after the last reading.

The exported `*_predictions.csv` files provide the same interface for the held-out trips.
Columns are `trip_id, time_s, serving_cell, target_time_s, p_cell_0, p_cell_1, p_cell_2, p_cell_3`.
CSV playback lets other modules use the predictions directly.


In [23]:
keys = ["time_s", "x_m", "y_m", "speed_m_s", "heading_rad", "serving_cell"]
readings = [dict(zip(keys, row), trip_id=trip_id) for row in trips[trip_id][:5]]
predictor = Predictor(output_dir / f"finetuned_{adaptation_trips}.pt")
print("Probabilities for cells 0 through 3", predictor.predict_proba(readings))


Probabilities for cells 0 through 3 [9.9999332e-01 3.4853397e-06 3.2692374e-06 2.7351550e-08]


## Regression checks

These checks cover trip boundaries, future-label leakage, adaptation budgets, checkpoint
reload, probability output, invalid readings, and repeatable training on a small dataset.


In [24]:
def trip(offset=0):
    return [[t, offset + t, 20, 1.2, 0, 0 if t < 7 else 1] for t in range(12)]


In [25]:
class TrainingTests(unittest.TestCase):
    def test_windows_stay_in_one_trip_and_target_t_plus_three(self):
        windows = make_windows({0: trip(), 1: trip(100)})
        self.assertEqual(windows.x.shape, (10, 5, 9))
        np.testing.assert_array_equal(windows.trip_ids, [0] * 5 + [1] * 5)
        np.testing.assert_array_equal(windows.times, [4, 5, 6, 7, 8] * 2)
        np.testing.assert_array_equal(windows.y, [1] * 10)
        np.testing.assert_allclose(windows.x[5, :, 0], np.arange(100, 105) / 400)
        np.testing.assert_array_equal(windows.current, [0, 0, 0, 1, 1] * 2)

    def test_future_rows_do_not_enter_input(self):
        original = trip()
        modified = trip()
        modified[7] = [7, 999, 999, 0, 3, 2]
        before = make_windows({0: original})
        after = make_windows({0: modified})
        torch.testing.assert_close(before.x[0], after.x[0])
        self.assertEqual(before.y[0].item(), 1)
        self.assertEqual(after.y[0].item(), 2)

    def test_nested_whole_trip_budgets(self):
        trips = {i: trip(i * 20) for i in range(50)}
        for count in (10, 25, 50):
            self.assertEqual(set(make_windows(trips, count).trip_ids), set(range(count)))
        with self.assertRaises(ValueError):
            make_windows(trips, 51)

    def test_checkpoint_and_online_probabilities_match(self):
        seed_everything(42)
        model = MobilityGRU(8)
        with tempfile.TemporaryDirectory() as directory:
            checkpoint = Path(directory) / "model.pt"
            save_checkpoint(model, checkpoint, 8, 42, {})
            predictor = Predictor(checkpoint)
            keys = ["time_s", "x_m", "y_m", "speed_m_s", "heading_rad", "serving_cell"]
            readings = [dict(zip(keys, row), trip_id=0) for row in trip()[:5]]
            online = predictor.predict_proba(readings)
            batch = probabilities(model, make_windows({0: trip()}).x)[0]
            np.testing.assert_allclose(online, batch, atol=1e-7)
            self.assertAlmostEqual(float(online.sum()), 1.0, places=6)
            self.assertTrue(np.all(online >= 0))
            readings[-1]["trip_id"] = 1
            with self.assertRaises(ValueError):
                predictor.predict_proba(readings)
            readings[-1]["trip_id"] = 0
            readings[-1]["time_s"] = 4.1
            with self.assertRaises(ValueError):
                predictor.predict_proba(readings)

    def test_changed_accuracy_exposes_persistence(self):
        windows = make_windows({0: trip()})
        scores = metrics(np.eye(4)[windows.current], windows)
        self.assertEqual(scores["changed_windows"], 3)
        self.assertEqual(scores["changed_accuracy"], 0)
        self.assertEqual(scores["unchanged_accuracy"], 1)

    def test_training_repeats_exactly(self):
        windows = make_windows({0: trip(), 1: trip(100)})
        states = []
        for _ in range(2):
            seed_everything(42)
            model = MobilityGRU(8)
            fit(model, windows, windows, 2, 0.001, 4, 2, 42)
            states.append(probabilities(model, windows.x))
        np.testing.assert_array_equal(*states)

    def test_invalid_readings(self):
        for rows in ([[1, 2]], [[0, 1, 2, 1, 0, 0.5]], [[0, float("nan"), 2, 1, 0, 0]]):
            with self.assertRaises(ValueError):
                encode(rows)

    def test_reader_rejects_missing_seconds_and_interleaving(self):
        header = "trip_id,time_s,x_m,y_m,speed_m_s,heading_rad,serving_cell\n"
        examples = ["0,0,0,0,1,0,0\n0,2,0,0,1,0,0\n",
                    "0,0,0,0,1,0,0\n1,0,0,0,1,0,0\n0,1,0,0,1,0,0\n"]
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "invalid.csv"
            for content in examples:
                path.write_text(header + content, encoding="utf-8")
                with self.assertRaises(ValueError):
                    read_trips(path)


In [26]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TrainingTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Regression checks failed"


test_changed_accuracy_exposes_persistence (__main__.TrainingTests.test_changed_accuracy_exposes_persistence) ... 

ok


test_checkpoint_and_online_probabilities_match (__main__.TrainingTests.test_checkpoint_and_online_probabilities_match) ... 

ok


test_future_rows_do_not_enter_input (__main__.TrainingTests.test_future_rows_do_not_enter_input) ... 

ok


... (9 outputs left out)

ok


test_windows_stay_in_one_trip_and_target_t_plus_three (__main__.TrainingTests.test_windows_stay_in_one_trip_and_target_t_plus_three) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 0.147s

OK
